### GOAL OF NOTEBOOK 06

We will build:
✅ function calling
✅ airline support tools
✅ agent routing logic
✅ tool execution pipeline

This is one of the MOST important notebooks.

In [1]:
# Imports
import json
import os

from dotenv import load_dotenv
from openai import OpenAI

In [2]:
# Load Environment Variables
load_dotenv("../.env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [3]:
# Initialize Client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

MODEL = "openai/gpt-oss-120b:free"

print("LLM initialized!")

LLM initialized!


In [4]:
# Create Airline Support Functions
def check_baggage_policy(airline):

    policies = {
        "Delta": "Delta allows 1 checked bag up to 23kg for international economy flights.",
        "United": "United baggage policy allows one cabin bag and checked baggage fees may apply.",
        "American": "American Airlines charges for checked baggage depending on route and fare."
    }

    return policies.get(
        airline,
        "Baggage policy not found."
    )

In [5]:
# Compensation Function
def calculate_compensation(delay_hours):

    if delay_hours >= 5:
        return "Passenger may be eligible for hotel, meals, and compensation."
    
    elif delay_hours >= 3:
        return "Passenger may be eligible for meal vouchers."
    
    else:
        return "No compensation available."

In [6]:
# Flight Status Function
def check_flight_status(flight_number):

    fake_database = {
        "AI202": "Delayed by 2 hours",
        "DL404": "Boarding",
        "UA101": "Cancelled"
    }

    return fake_database.get(
        flight_number,
        "Flight not found."
    )

In [7]:
# Define Tools for LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_baggage_policy",
            "description": "Get airline baggage policy",
            "parameters": {
                "type": "object",
                "properties": {
                    "airline": {
                        "type": "string"
                    }
                },
                "required": ["airline"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "calculate_compensation",
            "description": "Calculate passenger compensation for delays",
            "parameters": {
                "type": "object",
                "properties": {
                    "delay_hours": {
                        "type": "integer"
                    }
                },
                "required": ["delay_hours"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "check_flight_status",
            "description": "Check current flight status",
            "parameters": {
                "type": "object",
                "properties": {
                    "flight_number": {
                        "type": "string"
                    }
                },
                "required": ["flight_number"]
            }
        }
    }
]

In [8]:
# First Function Calling Test
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "What compensation can I get for a 6 hour delay?"
        }
    ],
    tools=tools
)

response

ChatCompletion(id='gen-1779370716-Etu3FS6w4gBInguqsXRm', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-829397407ab50776', function=Function(arguments='{\n  "delay_hours": 6\n}', name='calculate_compensation'), type='function', index=0)], reasoning='We need to calculate compensation. Use calculate_compensation with delay_hours 6.', reasoning_details=[{'type': 'reasoning.text', 'text': 'We need to calculate compensation. Use calculate_compensation with delay_hours 6.', 'format': 'unknown', 'index': 0}]), native_finish_reason='tool_calls')], created=1779370717, model='openai/gpt-oss-120b:free', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=46, prompt_tokens=180, total_tokens=226, completion_tokens_details=Completi

In [9]:
# Extract Tool Call
tool_call = response.choices[0].message.tool_calls[0]

tool_name = tool_call.function.name

tool_args = json.loads(
    tool_call.function.arguments
)

print("Tool Name:", tool_name)
print("Arguments:", tool_args)

Tool Name: calculate_compensation
Arguments: {'delay_hours': 6}


In [10]:
# Execute Tool Automatically
if tool_name == "calculate_compensation":

    result = calculate_compensation(
        tool_args["delay_hours"]
    )

elif tool_name == "check_baggage_policy":

    result = check_baggage_policy(
        tool_args["airline"]
    )

elif tool_name == "check_flight_status":

    result = check_flight_status(
        tool_args["flight_number"]
    )

else:
    result = "Tool not found."

print(result)

Passenger may be eligible for hotel, meals, and compensation.


In [11]:
# Send Tool Result Back To LLM
messages = [
    {
        "role": "user",
        "content": "What compensation can I get for a 6 hour delay?"
    },

    response.choices[0].message.model_dump(),

    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result
    }
]

In [12]:
# Generate Final AI Response
final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools
)

print(
    final_response.choices[0]
    .message.content
)

For a 6‑hour delay you’re generally entitled to **EU Regulation 261/2004** compensation (if the flight is from, to, or operated by an EU carrier). The typical compensation amounts are:

| Flight distance | Compensation |
|----------------|--------------|
| Up to 1,500 km | €250 |
| 1,500‑3,500 km | €400 |
| Over 3,500 km | €600 |

In addition, airlines must provide **reasonable assistance** during the delay, which can include:

* **Meals and refreshments** (proportionate to the waiting time)  
* **Two free telephone calls, faxes or emails**  
* **Hotel accommodation and transport between the airport and hotel** (if the delay extends overnight)

If the airline cannot prove the delay was caused by extraordinary circumstances (e.g., severe weather, air‑traffic control strikes), they are liable for the above compensation.  
If the airline offers a **voucher or alternate travel arrangement**, you can accept it only if it meets or exceeds the statutory amount.

**Next steps**

1. **Confirm t